# 03. TB Sensitivity Deep Dive Analysis

**Objective:** Comprehensive analysis of Tuberculosis detection performance.

**Focus:**
- TB class performance metrics
- False negative analysis
- Confusion matrix breakdown
- Attention visualization for TB cases

---

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Results

In [ ]:
# Load metrics
metrics_file = Path('../results/metrics/all_metrics.json')

if not metrics_file.exists():
    print(f"❌ Metrics file not found: {metrics_file}")
    print("\nPlease run evaluation first:")
    print("  uv run python scripts/evaluate_models.py")
else:
    with open(metrics_file) as f:
        metrics = json.load(f)
    
    df = pd.DataFrame(metrics)
    print(f"✓ Loaded metrics for {len(df)} models")

## 2. TB-Specific Metrics Overview

In [ ]:
if 'df' in locals():
    # TB metrics table
    tb_metrics = df[['model_name', 'tb_sensitivity', 'tb_precision', 
                     'tb_f1', 'tb_specificity']].copy()
    
    # Sort by sensitivity
    tb_metrics = tb_metrics.sort_values('tb_sensitivity', ascending=False)
    
    # Format
    formatted_tb = tb_metrics.copy()
    for col in ['tb_sensitivity', 'tb_precision', 'tb_f1', 'tb_specificity']:
        formatted_tb[col] = formatted_tb[col].apply(lambda x: f"{x:.4f}")
    
    print("\n🫁 Tuberculosis Detection Metrics")
    print("=" * 80)
    print(formatted_tb.to_string(index=False))
    print("=" * 80)
    
    # Definitions
    print("\n📖 Metric Definitions:")
    print("  - Sensitivity (Recall): TP / (TP + FN) - Ability to detect TB cases")
    print("  - Precision: TP / (TP + FP) - Accuracy of TB predictions")
    print("  - F1-Score: Harmonic mean of Precision and Recall")
    print("  - Specificity: TN / (TN + FP) - Ability to identify non-TB cases")

## 3. TB Sensitivity Improvement Analysis

In [ ]:
if 'df' in locals():
    # Extract backbone and attention
    df['backbone'] = df['model_name'].apply(lambda x: 'resnet50' if 'resnet' in x else 'densenet121')
    df['attention'] = df['model_name'].apply(
        lambda x: 'none' if 'baseline' in x else 
                 'cbam' if 'cbam' in x else 
                 'se' if 'se' in x else 
                 'eca' if 'eca' in x else 'none'
    )
    
    # Calculate improvements
    resnet_baseline = df[(df['backbone'] == 'resnet50') & (df['attention'] == 'none')]['tb_sensitivity'].values
    resnet_cbam = df[(df['backbone'] == 'resnet50') & (df['attention'] == 'cbam')]['tb_sensitivity'].values
    
    if len(resnet_baseline) > 0 and len(resnet_cbam) > 0:
        baseline = resnet_baseline[0]
        cbam = resnet_cbam[0]
        absolute_improvement = cbam - baseline
        relative_improvement = (absolute_improvement / baseline) * 100
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Bar chart with improvement
        bars = ax.bar(['Baseline', '+CBAM'], [baseline, cbam], 
                     color=['#95a5a6', '#e74c3c'], alpha=0.8)
        
        # Add value labels
        for i, v in enumerate([baseline, cbam]):
            ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=14, fontweight='bold')
        
        # Add improvement annotation
        ax.annotate(f'+{absolute_improvement:.4f} ({relative_improvement:+.1f}%)',
                   xy=(1, cbam), xytext=(1.5, (baseline + cbam) / 2),
                   arrowprops=dict(arrowstyle='->', color='green', lw=2),
                   fontsize=12, color='green', fontweight='bold',
                   ha='center')
        
        ax.set_ylabel('TB Sensitivity', fontsize=12)
        ax.set_title('ResNet50: CBAM Impact on TB Detection', fontsize=14, fontweight='bold')
        ax.set_ylim(0, 1.1)
        
        plt.tight_layout()
        plt.savefig('../results/figures/08_tb_sensitivity_improvement.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n✓ Saved: ../results/figures/08_tb_sensitivity_improvement.png")

## 4. Confusion Matrix Analysis for TB

In [ ]:
if 'df' in locals() and 'confusion_matrix' in df.columns:
    class_names = ['Normal', 'Pneumonia', 'Tuberculosis', 'COVID-19']
    tb_idx = 2  # Tuberculosis index
    
    # Select models to compare
    models_to_show = ['resnet50_baseline', 'resnet50_cbam', 'densenet121_cbam']
    models_data = df[df['model_name'].isin(models_to_show)]
    
    if len(models_data) > 0:
        n_models = len(models_data)
        fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
        
        if n_models == 1:
            axes = [axes]
        
        for idx, (_, row) in enumerate(models_data.iterrows()):
            cm = np.array(row['confusion_matrix'])
            
            # Normalize by rows
            cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
            
            # Highlight TB row
            sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                       xticklabels=class_names, yticklabels=class_names,
                       ax=axes[idx], cbar=False)
            
            # Add TB sensitivity annotation
            tb_sensitivity = cm[tb_idx, tb_idx] / cm[tb_idx, :].sum() if cm[tb_idx, :].sum() > 0 else 0
            axes[idx].set_title(f"{row['model_name']}\nTB Sensitivity: {tb_sensitivity:.3f}",
                               fontsize=11, fontweight='bold')
        
        plt.suptitle('Confusion Matrices: TB Analysis', fontsize=14, fontweight='bold', y=1.05)
        plt.tight_layout()
        plt.savefig('../results/figures/09_tb_confusion_matrices.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n✓ Saved: ../results/figures/09_tb_confusion_matrices.png")

## 5. False Negative Analysis

In [ ]:
if 'df' in locals() and 'confusion_matrix' in df.columns:
    tb_idx = 2
    
    # Calculate false negative rate for TB
    fnr_data = []
    
    for _, row in df.iterrows():
        cm = np.array(row['confusion_matrix'])
        
        # TB false negatives = TB cases predicted as other classes
        tb_actual = cm[tb_idx, :].sum()
        tb_correct = cm[tb_idx, tb_idx]
        fn = tb_actual - tb_correct
        fnr = fn / tb_actual if tb_actual > 0 else 0
        
        fnr_data.append({
            'model_name': row['model_name'],
            'tb_actual': tb_actual,
            'tb_correct': tb_correct,
            'false_negatives': fn,
            'fnr': fnr
        })
    
    df_fnr = pd.DataFrame(fnr_data)
    df_fnr = df_fnr.sort_values('fnr', ascending=True)
    
    # Display
    print("\n🔴 False Negative Analysis for Tuberculosis")
    print("=" * 80)
    
    display_df = df_fnr[['model_name', 'tb_actual', 'tb_correct', 'false_negatives', 'fnr']].copy()
    display_df['fnr'] = display_df['fnr'].apply(lambda x: f"{x:.4f}")
    print(display_df.to_string(index=False))
    print("=" * 80)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    
    sns.barplot(data=df_fnr, x='model_name', y='fnr', ax=ax, palette='Reds_r')
    ax.set_ylabel('False Negative Rate (FNR)', fontsize=12)
    ax.set_xlabel('Model', fontsize=12)
    ax.set_title('TB False Negative Rate by Model (Lower is Better)', fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels
    for i, row in df_fnr.iterrows():
        ax.text(i, row['fnr'] + 0.01, f"{row['fnr']:.3f}", ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../results/figures/10_tb_false_negative_rate.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Saved: ../results/figures/10_tb_false_negative_rate.png")

## 6. Per-Class Recall Comparison

In [ ]:
if 'df' in locals() and 'per_class_recall' in df.columns:
    # Expand per-class recall
    recall_data = []
    class_names = ['Normal', 'Pneumonia', 'Tuberculosis', 'COVID-19']
    
    for _, row in df.iterrows():
        for i, class_name in enumerate(class_names):
            if i < len(row['per_class_recall']):
                recall_data.append({
                    'model_name': row['model_name'],
                    'class': class_name,
                    'recall': row['per_class_recall'][i]
                })
    
    df_recall = pd.DataFrame(recall_data)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Grouped bar chart
    sns.barplot(data=df_recall, x='model_name', y='recall', hue='class', ax=ax)
    ax.set_ylabel('Recall (Sensitivity)', fontsize=12)
    ax.set_xlabel('Model', fontsize=12)
    ax.set_title('Per-Class Recall Comparison', fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.savefig('../results/figures/11_per_class_recall.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Saved: ../results/figures/11_per_class_recall.png")

## 7. Clinical Impact Summary

In [ ]:
if 'df' in locals():
    print("\n" + "=" * 80)
    print("🏥 CLINICAL IMPACT SUMMARY")
    print("=" * 80)
    
    # Best model for TB
    best_tb_model = df.loc[df['tb_sensitivity'].idxmax()]
    
    print(f"\n🏆 Best Model for TB Detection: {best_tb_model['model_name']}")
    print(f"   - TB Sensitivity: {best_tb_model['tb_sensitivity']:.4f}")
    print(f"   - TB Precision: {best_tb_model['tb_precision']:.4f}")
    print(f"   - TB F1-Score: {best_tb_model['tb_f1']:.4f}")
    
    # Calculate lives impacted (hypothetical)
    print(f"\n📊 Hypothetical Clinical Impact (per 1000 TB patients):")
    
    baseline_model = df[df['model_name'].str.contains('baseline')].iloc[0]
    baseline_sensitivity = baseline_model['tb_sensitivity']
    best_sensitivity = best_tb_model['tb_sensitivity']
    
    baseline_detected = int(1000 * baseline_sensitivity)
    best_detected = int(1000 * best_sensitivity)
    additional_detected = best_detected - baseline_detected
    
    print(f"   - Baseline model detects: {baseline_detected}/1000 TB cases")
    print(f"   - Best model detects: {best_detected}/1000 TB cases")
    print(f"   - ✨ Additional cases detected: +{additional_detected}/1000")
    
    print(f"\n📁 All figures saved to: ../results/figures/")
    print("=" * 80)